In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

db_path = "../data/Chinook_Sqlite.sqlite"
export_dir = Path("../data/chinook_exports")
export_dir.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(db_path)

In [2]:
from IPython.display import display
import numpy as np


In [3]:
tracks_query = """
SELECT
    t.TrackId AS track_id,
    t.Name AS track_name,
    al.Title AS album_title,
    ar.Name AS artist_name,
    g.Name AS genre_name,
    mt.Name AS media_type,
    t.Milliseconds AS milliseconds,
    t.UnitPrice AS unit_price
FROM Track t
JOIN Album al
    ON t.AlbumId = al.AlbumId
JOIN Artist ar
    ON al.ArtistId = ar.ArtistId
JOIN Genre g
    ON t.GenreId = g.GenreId
JOIN MediaType mt
    ON t.MediaTypeId = mt.MediaTypeId;
"""

tracks_df = pd.read_sql_query(tracks_query, conn)
tracks_df.to_csv(export_dir / "tracks.csv", index=False)

In [4]:
customers_query = """
SELECT
    CustomerId AS customer_id,
    FirstName AS first_name,
    LastName AS last_name,
    Company AS company,
    City AS city,
    State AS state,
    Country AS country,
    Email AS email
FROM Customer;
"""

customers_df = pd.read_sql_query(customers_query, conn)
customers_df.to_csv(export_dir / "customers.csv", index=False)

In [5]:
invoice_items_query = """
SELECT
    il.InvoiceLineId AS invoice_line_id,
    il.InvoiceId AS invoice_id,
    i.CustomerId AS customer_id,
    i.InvoiceDate AS invoice_date,
    i.BillingCountry AS billing_country,
    il.TrackId AS track_id,
    il.UnitPrice AS unit_price,
    il.Quantity AS quantity
FROM InvoiceLine il
JOIN Invoice i
    ON il.InvoiceId = i.InvoiceId;
"""

invoice_items_df = pd.read_sql_query(invoice_items_query, conn)
invoice_items_df.to_csv(export_dir / "invoice_items.csv", index=False)

conn.close()

In [6]:
## Part 1 — Load, Inspect, and Prepare
tracks = pd.read_csv("../data/chinook_exports/tracks.csv")
customers = pd.read_csv("../data/chinook_exports/customers.csv")
invoice_items = pd.read_csv("../data/chinook_exports/invoice_items.csv")

In [7]:
print("Shape: " ,tracks.shape)
print(tracks.info())
print(tracks.describe())
tracks.head()


Shape:  (3503, 8)
<class 'pandas.DataFrame'>
RangeIndex: 3503 entries, 0 to 3502
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   track_id      3503 non-null   int64  
 1   track_name    3503 non-null   str    
 2   album_title   3503 non-null   str    
 3   artist_name   3503 non-null   str    
 4   genre_name    3503 non-null   str    
 5   media_type    3503 non-null   str    
 6   milliseconds  3503 non-null   int64  
 7   unit_price    3503 non-null   float64
dtypes: float64(1), int64(2), str(5)
memory usage: 219.1 KB
None
          track_id  milliseconds   unit_price
count  3503.000000  3.503000e+03  3503.000000
mean   1752.000000  3.935992e+05     1.050805
std    1011.373324  5.350054e+05     0.239006
min       1.000000  1.071000e+03     0.990000
25%     876.500000  2.072810e+05     0.990000
50%    1752.000000  2.556340e+05     0.990000
75%    2627.500000  3.216450e+05     0.990000
max    3503.000000  5.28

,track_id,track_name,album_title,artist_name,genre_name,media_type,milliseconds,unit_price
0,1,For Those About To Rock (We Salute You),For Those About To Rock We Salute You,AC/DC,Rock,MPEG audio file,343719,0.99
1,2,Balls to the Wall,Balls to the Wall,Accept,Rock,Protected AAC audio file,342562,0.99
2,3,Fast As a Shark,Restless and Wild,Accept,Rock,Protected AAC audio file,230619,0.99
3,4,Restless and Wild,Restless and Wild,Accept,Rock,Protected AAC audio file,252051,0.99
4,5,Princess of the Dawn,Restless and Wild,Accept,Rock,Protected AAC audio file,375418,0.99


In [8]:
print("Shape: " ,customers.shape)
print(customers.info())
print(customers.describe())
customers.head()

Shape:  (59, 8)
<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  59 non-null     int64
 1   first_name   59 non-null     str  
 2   last_name    59 non-null     str  
 3   company      10 non-null     str  
 4   city         59 non-null     str  
 5   state        30 non-null     str  
 6   country      59 non-null     str  
 7   email        59 non-null     str  
dtypes: int64(1), str(7)
memory usage: 3.8 KB
None
       customer_id
count    59.000000
mean     30.000000
std      17.175564
min       1.000000
25%      15.500000
50%      30.000000
75%      44.500000
max      59.000000


,customer_id,first_name,last_name,company,city,state,country,email
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,São José dos Campos,SP,Brazil,luisg@embraer.com.br
1,2,Leonie,Köhler,NaN,Stuttgart,NaN,Germany,leonekohler@surfeu.de
2,3,François,Tremblay,NaN,Montréal,QC,Canada,ftremblay@gmail.com
3,4,Bjørn,Hansen,NaN,Oslo,NaN,Norway,bjorn.hansen@yahoo.no
4,5,František,Wichterlová,JetBrains s.r.o.,Prague,NaN,Czech Republic,frantisekw@jetbrains.com


In [9]:
print("Shape: " ,invoice_items.shape)
print(invoice_items.info())
print(invoice_items.describe())
invoice_items.head()

Shape:  (2240, 8)
<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   invoice_line_id  2240 non-null   int64  
 1   invoice_id       2240 non-null   int64  
 2   customer_id      2240 non-null   int64  
 3   invoice_date     2240 non-null   str    
 4   billing_country  2240 non-null   str    
 5   track_id         2240 non-null   int64  
 6   unit_price       2240 non-null   float64
 7   quantity         2240 non-null   int64  
dtypes: float64(1), int64(5), str(2)
memory usage: 140.1 KB
None
       invoice_line_id   invoice_id  customer_id     track_id   unit_price  \
count      2240.000000  2240.000000  2240.000000  2240.000000  2240.000000   
mean       1120.500000   206.868750    29.974107  1717.734375     1.039554   
std         646.776623   119.134877    17.018715   993.797999     0.217069   
min           1.000000     1.000000     1.000000  

,invoice_line_id,invoice_id,customer_id,invoice_date,billing_country,track_id,unit_price,quantity
0,1,1,2,2009-01-01 00:00:00,Germany,2,0.99,1
1,2,1,2,2009-01-01 00:00:00,Germany,4,0.99,1
2,3,2,4,2009-01-02 00:00:00,Norway,6,0.99,1
3,4,2,4,2009-01-02 00:00:00,Norway,8,0.99,1
4,5,2,4,2009-01-02 00:00:00,Norway,10,0.99,1


In [10]:
# Converting invoice_date to datetime
invoice_items["invoice_date"] = pd.to_datetime(invoice_items["invoice_date"], format = "%Y-%m-%d %H:%M:%S")

In [11]:
# Add columns
tracks["minutes"] = tracks["milliseconds"] / 60000.0
invoice_items["revenue"] = invoice_items["quantity"] * invoice_items["unit_price"]
invoice_items["invoice_year"] = invoice_items["invoice_date"].dt.year
invoice_items["invoice_month"] = invoice_items["invoice_date"].dt.month

print(tracks["minutes"].head())
invoice_items.loc[0:5, ["revenue", "invoice_year","invoice_month"]]

# tracks_invoice_merged = pd.merge(tracks, invoice_items, on ="track_id", how = "left")
# tracks["minutes"] = tracks_invoice_merged["invoice_date"].dt.minute
# tracks["minutes"].head()

0    5.728650
1    5.709367
2    3.843650
3    4.200850
4    6.256967
Name: minutes, dtype: float64


,revenue,invoice_year,invoice_month
0,0.99,2009,1
1,0.99,2009,1
2,0.99,2009,1
3,0.99,2009,1
4,0.99,2009,1
5,0.99,2009,1


In [12]:
## Part 2 — Filtering and Indexing
special_tracks  = tracks[(tracks["genre_name"] == "Rock") & (tracks["minutes"] > 5) & (tracks["unit_price"] > 0.8)]
special_tracks.loc[: ,["track_name", "artist_name", "album_title", "genre_name", "minutes", "unit_price"]].head()

,track_name,artist_name,album_title,genre_name,minutes,unit_price
0,For Those About To Rock (We Salute You),AC/DC,For Those About To Rock We Salute You,Rock,5.728650,0.99
1,Balls to the Wall,Accept,Balls to the Wall,Rock,5.709367,0.99
4,Princess of the Dawn,Accept,Restless and Wild,Rock,6.256967,0.99
14,Go Down,AC/DC,Let There Be Rock,Rock,5.519667,0.99
16,Let There Be Rock,AC/DC,Let There Be Rock,Rock,6.110900,0.99


In [13]:
special_tracks.sort_values(by = "minutes", ascending= False).head(10)

,track_id,track_name,album_title,artist_name,genre_name,media_type,milliseconds,unit_price,minutes
1665,1666,Dazed And Confused,The Song Remains The Same (Disc 1),Led Zeppelin,Rock,MPEG audio file,1612329,0.99,26.872150
619,620,Space Truckin',The Final Concerts (Disc 2),Deep Purple,Rock,MPEG audio file,1196094,0.99,19.934900
1580,1581,Dazed And Confused,BBC Sessions [Disc 2] [Live],Led Zeppelin,Rock,MPEG audio file,1116734,0.99,18.612233
2428,2429,We've Got To Get Together/Jingo,Santana Live,Santana,Rock,MPEG audio file,1070027,0.99,17.833783
2431,2432,Funky Piano,Santana Live,Santana,Rock,MPEG audio file,934791,0.99,15.579850
620,621,Going Down / Highway Star,The Final Concerts (Disc 2),Deep Purple,Rock,MPEG audio file,913658,0.99,15.227633
2426,2427,Santana Jam,Santana - As Years Go By,Santana,Rock,MPEG audio file,882834,0.99,14.713900
2564,2565,The Sun Road,[1997] Black Light Syndrome,"Terry Bozzio, Tony Levin & Steve Stevens",Rock,MPEG audio file,880640,0.99,14.677333
1669,1670,Whole Lotta Love,The Song Remains The Same (Disc 2),Led Zeppelin,Rock,MPEG audio file,863895,0.99,14.398250
621,622,Mistreated (Alternate Version),The Final Concerts (Disc 2),Deep Purple,Rock,MPEG audio file,854700,0.99,14.245000


In [14]:
## Part 3 — Adding and Modifying Columns

conditions = [ tracks["minutes"] < 3,
              ((6>tracks["minutes"]) & (tracks["minutes"]> 3)),
                tracks["minutes"]>6]
choices = ["Short", "Medium", "Long"]
tracks["duration_category"] = np.select(conditions, choices, default = "false")
print(tracks["duration_category"].value_counts())


# or using .loc
tracks["category"] = "init"
tracks.loc[tracks["minutes"] < 3 , "category"] = "Short" 
tracks.loc[((6>tracks["minutes"]) & (tracks["minutes"]> 3)), "category"] = "Medium"
tracks.loc[tracks["minutes"]>6 , "category"] = "Long"
print(tracks["category"].value_counts()) 




duration_category
Medium    2400
Long       623
Short      480
Name: count, dtype: int64
category
Medium    2400
Long       623
Short      480
Name: count, dtype: int64


In [15]:
tracks["price_category"] = "init"
tracks.loc[tracks["unit_price"] > 0.99 , "price_category"] = "Premium"
tracks.loc[tracks["unit_price"] == 0.99 , "price_category"] = "Standard"
print(tracks["price_category"].value_counts())

price_category
Standard    3290
Premium      213
Name: count, dtype: int64


In [16]:
## Part 4 — GroupBy and Aggregation
grouped_tracks = tracks.groupby("genre_name")
genre_summary = grouped_tracks.agg(
    track_count = ("track_id", "count"),
    avg_duration = ("minutes", "mean"),
    max_duration = ("minutes", "max"),
    avg_unit_price = ("unit_price", "mean")
)
sorted_genre_summary = genre_summary.sort_values("track_count", ascending = False)

cleaned = sorted_genre_summary.reset_index()
cleaned.head()


,genre_name,track_count,avg_duration,max_duration,avg_unit_price
0,Rock,1297,4.731834,26.872150,0.99
1,Latin,579,3.880988,9.050117,0.99
2,Metal,374,5.162491,13.608483,0.99
3,Alternative & Punk,332,3.905897,9.310033,0.99
4,Jazz,130,4.862590,15.125333,0.99


In [17]:
## Part 5 — Pivot Table

merged = pd.merge(invoice_items,tracks, on = "track_id", how = "left")
pivot_df = pd.pivot_table(data = merged,
                          values = "revenue",
                          index = "billing_country",
                          columns = "genre_name",
                          aggfunc = "sum")

pivot_df.fillna(value = 0, inplace = True)
pivot_df["total_revenue"] = pivot_df.sum(axis = 1)   # Returns a Series with the sum of each row
pivot_df["total_revenue"]
pivot_df.sort_values("total_revenue",ascending = False, inplace= True)
display(pivot_df)


genre_name,Alternative,Alternative & Punk,Blues,Bossa Nova,Classical,Comedy,Drama,Easy Listening,Electronica/Dance,Heavy Metal,...,R&B/Soul,Reggae,Rock,Rock And Roll,Sci Fi & Fantasy,Science Fiction,Soundtrack,TV Shows,World,total_revenue
billing_country,,,,,,,,,,,,,,,,,,,,,
USA,4.95,49.50,14.85,6.93,7.92,15.92,11.94,2.97,0.00,3.96,...,11.88,5.94,155.43,2.97,9.95,1.99,3.96,27.86,0.00,523.06
Canada,0.00,35.64,3.96,6.93,4.95,0.00,3.98,0.00,3.96,0.00,...,4.95,6.93,105.93,1.98,0.00,0.00,0.00,1.99,5.94,303.96
France,3.96,30.69,1.98,0.99,9.90,0.00,7.96,0.00,1.98,0.00,...,0.00,0.99,64.35,0.99,3.98,0.00,4.95,1.99,0.00,195.10
Brazil,0.00,6.93,5.94,0.00,5.94,0.00,0.00,0.00,0.00,0.00,...,2.97,5.94,80.19,0.00,3.98,0.00,3.96,0.00,1.98,190.10
Germany,0.99,12.87,13.86,0.00,0.00,0.00,1.99,1.98,0.00,2.97,...,0.00,0.00,61.38,0.00,0.00,3.98,4.95,5.97,0.00,156.48
United Kingdom,0.00,8.91,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,1.98,4.95,36.63,0.00,0.00,0.00,0.00,0.00,0.99,112.86
Czech Republic,0.00,8.91,0.99,0.00,0.00,0.00,11.94,0.00,1.98,0.00,...,1.98,0.00,24.75,0.00,0.00,1.99,0.00,15.92,0.00,90.24
Portugal,0.00,4.95,0.99,0.00,0.00,0.00,0.00,0.00,0.99,1.98,...,1.98,0.00,30.69,0.00,1.99,0.00,0.00,1.99,1.98,77.24
India,0.00,10.89,2.97,0.00,1.98,0.00,0.00,0.00,0.00,0.00,...,3.96,0.00,24.75,0.00,1.99,0.00,0.00,1.99,0.00,75.26


In [18]:
## Part 6 — Merge Practice
merge1 = pd.merge(invoice_items, customers, on = "customer_id", how = "left")
merge2 = pd.merge(merge1, tracks, on = "track_id", how = "left")

selected_df = merge2.loc[:, [
    "invoice_id",
    "invoice_date",
    "customer_id",
    "first_name",
    "last_name",
    "country",
    "track_name",
    "artist_name",
    "genre_name",
    "quantity",
    "unit_price_x",
    "revenue"
]]
selected_df.rename(columns = {"unit_price_x":"unit_price"} , inplace= True)
selected_df["full_name"]  = selected_df["first_name"] + " " + selected_df["last_name"]

top_customers = (
    selected_df.groupby(["customer_id", "full_name"], as_index=False)["revenue"]
    .sum()
    .rename(columns={"revenue": "total_revenue"})
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

display(top_customers)
top_artists = (
    selected_df.groupby("artist_name", as_index=False)["revenue"]
    .sum()
    .rename(columns={"revenue": "total_revenue"})
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

display(top_artists)

,customer_id,full_name,total_revenue
5,6,Helena Holý,49.62
25,26,Richard Cunningham,47.62
56,57,Luis Rojas,46.62
44,45,Ladislav Kovács,45.62
45,46,Hugh O'Reilly,45.62
27,28,Julia Barnett,43.62
36,37,Fynn Zimmermann,43.62
23,24,Frank Ralston,43.62
6,7,Astrid Gruber,42.62
24,25,Victor Stevens,42.62


,artist_name,total_revenue
69,Iron Maiden,138.60
156,U2,105.93
95,Metallica,90.09
82,Led Zeppelin,86.13
85,Lost,81.59
147,The Office,49.75
109,Os Paralamas Do Sucesso,44.55
37,Deep Purple,43.56
51,Faith No More,41.58
49,Eric Clapton,39.60


In [19]:
## Part 7 — Missing Data Cleanup
customers.isna().sum()
customers_clean = customers.copy()
customers_clean[["company", "state"]] = customers_clean[["company", "state"]].fillna(value = {"company": "Individual", "state": "Unknow"})
customers_clean.isna().sum()

# I chose to fill the missing values instead of dropping rows because company and state are not critical columns.
# Dropping rows would remove customer records, but filling them with default values keeps the dataset usable

customer_id    0
first_name     0
last_name      0
company        0
city           0
state          0
country        0
email          0
dtype: int64

In [20]:
## Part 8 — Concat Practice
invoice_before = invoice_items[invoice_items["invoice_date"] < "2012"]
invoice_after = invoice_items[invoice_items["invoice_date"] >= "2012"]
concatted = pd.concat([invoice_before,invoice_after], ignore_index= True)

print(concatted.shape == invoice_items.shape)

True


## Part 9 — Pandas groupby() vs SQL GROUP BY

Pandas groupby() is similar to SQL GROUP BY because both split data into groups and calculate summaries for each group. In SQL, we usually write SELECT, GROUP BY, and aggregate functions such as COUNT(), SUM(), or AVG(). In Pandas, we use groupby(), select the column we want to summarize, and then apply aggregation methods such as mean(), sum(), count(), or agg(). The main difference is that SQL returns a query result table, while Pandas keeps the result as a Series or DataFrame that can be further modified, sorted, filtered, reset with reset_index(), or used in another analysis step.


## Part 10 — Pandas merge() vs SQL JOIN

Pandas merge() is similar to SQL JOIN because both combine two tables using a common key column. In Pandas, pd.merge(left, right, on="column") works like joining two SQL tables with JOIN ... ON column. The key column connects matching rows from the two DataFrames. Also, how="inner" is like an SQL INNER JOIN, where only matching rows from both DataFrames are kept, while how="left" is like an SQL LEFT JOIN, where all rows from the left DataFrame are kept and matching values from the right DataFrame are added when available.